In [1]:
%pip install transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [24]:
import sys
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer,AutoModel
sys.path.append(os.path.abspath(".."))
from shared import config
import torch



In [3]:
hate_df = pd.read_csv(config.PROCESSED_HATE_SPEECH_PATH)
hatexplain_df = pd.read_csv(config.PROCESSED_HATEXPLAIN_PATH)

print("Processed Hate Speech Dataset :", hate_df.shape)
print("Processed HateXplain Dataset :", hatexplain_df.shape)

Processed Hate Speech Dataset : (135556, 147)
Processed HateXplain Dataset : (20148, 7)


In [5]:
print("Hate Speech Dataset")
hate_df.head()

Hate Speech Dataset


,comment_id,annotator_id,platform,sentiment,respect,insult,humiliate,status,dehumanize,violence,...,annotator_religion_nothing,annotator_religion_other,annotator_sexuality_bisexual,annotator_sexuality_gay,annotator_sexuality_straight,annotator_sexuality_other,label,clean_text,text_length,word_count
0,47777,10873,3,0.0,0.0,0.0,0.0,2.0,0.0,0.0,...,False,False,False,False,True,False,normal,yes indeed she sort of reminds me of the elder...,239,51
1,39773,2790,2,0.0,0.0,0.0,0.0,2.0,0.0,0.0,...,False,False,False,False,True,False,normal,the trans women reading this tweet right now i...,57,10
2,47101,3379,3,4.0,4.0,4.0,4.0,4.0,4.0,0.0,...,True,False,False,False,True,False,hate,question these 4 broads who criticize america ...,231,43
3,43625,7365,3,2.0,3.0,2.0,1.0,2.0,0.0,0.0,...,False,False,False,False,True,False,normal,it is about time for all illegals to go back t...,123,24
4,12538,488,0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,False,False,False,False,True,False,hate,for starters bend over the one in pink and kic...,122,26


In [6]:
print("HateXplain Dataset")
hatexplain_df.head()

HateXplain Dataset


,post_id,text,labels,label,clean_text,text_length,word_count
0,1179055004553900032_twitter,i dont think im getting my baby them white 9 h...,"('normal', 'normal', 'normal')",normal,i dont think im getting my baby them white 9 h...,90,20
1,1179063826874032128_twitter,we cannot continue calling ourselves feminists...,"('normal', 'normal', 'normal')",normal,we cannot continue calling ourselves feminists...,247,42
2,1178793830532956161_twitter,nawt yall niggers ignoring me,"('normal', 'normal', 'hatespeech')",normal,nawt yall niggers ignoring me,29,5
3,1179088797964763136_twitter,<user> i am bit confused coz chinese ppl can n...,"('hatespeech', 'offensive', 'hatespeech')",hate,user i am bit confused coz chinese ppl can not...,116,23
4,1179085312976445440_twitter,this bitch in whataburger eating a burger with...,"('hatespeech', 'hatespeech', 'offensive')",hate,this bitch in whataburger eating a burger with...,101,20


In [7]:
common_columns = list(set(hate_df.columns).intersection(hatexplain_df.columns))
print(common_columns)

['label', 'text', 'clean_text', 'text_length', 'word_count']


In [8]:
combined_df = pd.concat(
    [
        hate_df[common_columns],
        hatexplain_df[common_columns]
    ],
    ignore_index=True
)

print("Combined Dataset Shape :", combined_df.shape)

Combined Dataset Shape : (155704, 5)


In [9]:
combined_df.info()
print(combined_df.head())

<class 'pandas.DataFrame'>
RangeIndex: 155704 entries, 0 to 155703
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   label        155704 non-null  str  
 1   text         155704 non-null  str  
 2   clean_text   155696 non-null  str  
 3   text_length  155704 non-null  int64
 4   word_count   155704 non-null  int64
dtypes: int64(2), str(3)
memory usage: 49.6 MB
    label                                               text  \
0  normal  Yes indeed. She sort of reminds me of the elde...   
1  normal  The trans women reading this tweet right now i...   
2    hate  Question: These 4 broads who criticize America...   
3  normal  It is about time for all illegals to go back t...   
4    hate  For starters bend over the one in pink and kic...   

                                          clean_text  text_length  word_count  
0  yes indeed she sort of reminds me of the elder...          239          51  
1  the trans women rea

In [10]:
print(combined_df["label"].value_counts())

label
normal       89357
hate         60867
offensive     5480
Name: count, dtype: int64


In [11]:
train_df, temp_df = train_test_split(combined_df,test_size=0.20,random_state=config.RANDOM_SEED,stratify=combined_df["label"])
valid_df, test_df = train_test_split(temp_df,test_size=0.50,random_state=config.RANDOM_SEED,stratify=temp_df["label"])

In [12]:
print("Training Samples :", len(train_df))
print("Validation Samples :", len(valid_df))
print("Testing Samples :", len(test_df))

Training Samples : 124563
Validation Samples : 15570
Testing Samples : 15571


In [13]:
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
print("Tokenizer Loaded Successfully")

Tokenizer Loaded Successfully


In [14]:
import shared.config as config

print(dir(config))

['BASE_DIR', 'BATCH_SIZE', 'DATA_DIR', 'HATEXPLAIN_DATASET_PATH', 'HATE_SPEECH_DATASET_PATH', 'LABEL_COLUMN', 'LEARNING_RATE', 'MAX_LENGTH', 'MODEL_NAME', 'NUM_EPOCHS', 'PROCESSED_DATA_DIR', 'PROCESSED_HATEXPLAIN_PATH', 'PROCESSED_HATE_SPEECH_PATH', 'Path', 'RANDOM_SEED', 'RAW_DATA_DIR', 'TEXT_COLUMN', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'os']


In [15]:
label_mapping = {label: idx for idx, label in enumerate(sorted(combined_df["label"].unique()))}

print(label_mapping)

{'hate': 0, 'normal': 1, 'offensive': 2}


In [16]:
train_df["label_id"] = train_df["label"].map(label_mapping)
valid_df["label_id"] = valid_df["label"].map(label_mapping)
test_df["label_id"] = test_df["label"].map(label_mapping)

In [17]:
class HateSpeechDataset(Dataset):

    def __init__(self, dataframe):
        self.texts = dataframe["clean_text"].tolist()
        self.labels = dataframe["label_id"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = tokenizer(self.texts[index],truncation=True,padding="max_length",max_length=config.MAX_LENGTH,return_tensors="pt")
        
        return {"input_ids": encoding["input_ids"].squeeze(0),"attention_mask": encoding["attention_mask"].squeeze(0),"label":torch.tensor(self.labels[index],dtype=torch.long)}

In [18]:
train_dataset = HateSpeechDataset(train_df)
valid_dataset = HateSpeechDataset(valid_df)
test_dataset = HateSpeechDataset(test_df)

In [19]:
train_loader = DataLoader(train_dataset,batch_size=config.BATCH_SIZE,shuffle=True)

valid_loader = DataLoader(valid_dataset,batch_size=config.BATCH_SIZE)
test_loader = DataLoader(test_dataset,batch_size=config.BATCH_SIZE)

In [20]:
modernbert = AutoModel.from_pretrained(config.MODEL_NAME)
print("ModernBERT Loaded Successfully")

C:\Users\tngur\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tngur\.cache\huggingface\hub\models--answerdotai--ModernBERT-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100

ModernBERT Loaded Successfully


In [22]:
class ModernBERTClassifier(nn.Module):

    def __init__(self, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(config.MODEL_NAME)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.bert.config.hidden_size,num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids,attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0]
        cls_embedding = self.dropout(cls_embedding)
        logits = self.classifier(cls_embedding)

        return logits

In [23]:
num_labels = len(label_mapping)
model = ModernBERTClassifier(num_labels)
print(model)

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 2861.94it/s]
[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ModernBERTClassifier(
  (bert): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=False)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=False)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(

In [25]:
batch = next(iter(train_loader))
input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]
with torch.no_grad():
    logits = model(input_ids=input_ids,attention_mask=attention_mask)

print("Logits Shape :", logits.shape)

Logits Shape : torch.Size([16, 3])


In [27]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(),lr=config.LEARNING_RATE)
print("Loss Function and Optimizer Created")

Loss Function and Optimizer Created


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Using Device :", device)

Using Device : cpu


In [32]:
print("Combined Dataset :", len(combined_df))
print("Training Samples :", len(train_df))
print("Validation Samples :", len(valid_df))
print("Testing Samples :", len(test_df))

print("\nModernBERT Loaded")
print("Tokenizer Ready")
print("Dataset Ready")
print("DataLoader Ready")
print("Model ready for training")

Combined Dataset : 155704
Training Samples : 124563
Validation Samples : 15570
Testing Samples : 15571

ModernBERT Loaded
Tokenizer Ready
Dataset Ready
DataLoader Ready
Model ready for training
